# Debugging Snowflake Performance Using Ibis and BigQuery

In [196]:
from ibis.interactive import *
import altair as alt
import pandas as pd

In [172]:
con = ibis.bigquery.connect(project_id="ibis-gbq", dataset_id="workflows")

In [173]:
t = (
    con.tables.jobs[_.name.startswith("Snowflake")]
    .select(
        "name",
        "started_at",
        "completed_at",
        duration_minutes=(_.completed_at.epoch_seconds() - _.started_at.epoch_seconds()) / 60.0,
    )
)

In [174]:
fast = t[t.started_at.date() < "2023-05-16"].order_by("started_at")
slow = t[t.started_at.date() >= "2023-05-16"].order_by("started_at")

In [175]:
fast_source = fast.to_pandas()

In [176]:
slow_source = slow.to_pandas()

In [177]:
fast_chart = alt.Chart(fast_source)
slow_chart = alt.Chart(slow_source)

In [203]:
x = alt.X("yearmonthdate(started_at):T").title("Date")
y = alt.Y("ci1(duration_minutes):Q").title("Upper CI Bound of Duration (minutes)")

In [288]:
fc = fast_chart.mark_point().encode(x=x, y=y)
fc += fc.transform_loess("started_at", "duration_minutes").mark_line(color="red")

sc = slow_chart.mark_point().encode(x=x, y=y)
sc += sc.transform_loess("started_at", "duration_minutes").mark_line(color="red")

fc += sc

fc += alt.Chart(pd.DataFrame({"date": [pd.Timestamp("2023-05-16")]})).mark_rule().encode(x="yearmonthdate(date)")
df = pd.DataFrame({"x": [pd.Timestamp("2023-05-16")], "y": [29], "label": ["2023-05-16"]})
label = alt.Chart(df).mark_text(dy=10, dx=-35).encode(x="yearmonthdate(x):T", y="y:Q", text="label")
fc += label

fc.properties(width=1000).interactive()

alt.LayerChart(...)